# 02 迁移学习骨干网络比较（E1）

**目标：** 在固定划分、随机种子、增强和训练预算下，对比 ResNet18、MobileNetV2、EfficientNet-B0。

**前置条件：** Notebook 01 的标签审核、分组及四份固定清单已完成；已安装 `requirements.txt`。

**输入：** `configs/base.yaml`、`configs/experiments.yaml`、`configs/labels.yaml`、固定的 PlantVillage 训练与验证清单。

**输出：** `outputs/runs/e1_*/` 中的最佳权重、实际配置、环境记录和逐轮训练记录。本阶段使用验证集选择骨干；外部测试留到 Notebook 05。

默认 `RUN_TRAINING=False`，从头运行只展示实验计划，不下载预训练权重。

In [2]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

def summarize_validation(plan_list):
    records = []
    for plan in plan_list:
        run_dir = PROJECT_ROOT / "outputs" / "runs" / plan["run_id"]
        history = read_csv(run_dir / "history.csv")
        if not history:
            continue
        config_path = run_dir / "config.yaml"
        actual_config = load_config(config_path) if config_path.exists() else plan["config"]
        monitor = actual_config["training"].get("monitor", "macro_f1")
        key = "val_" + monitor
        best = min(history, key=lambda row: float(row[key])) if monitor == "loss" else max(
            history, key=lambda row: float(row[key]))
        records.append({"run_id": plan["run_id"], "recorded_epochs": len(history),
                        "monitor": monitor, "best_epoch": int(best["epoch"]),
                        "val_loss": float(best["val_loss"]),
                        "val_accuracy": float(best["val_accuracy"]),
                        "val_macro_f1": float(best["val_macro_f1"]),
                        "total_training_seconds": float(history[-1]["elapsed_seconds"])})
    return records


项目目录： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project
已配置类别数： 5


## 1. 建立实验计划

E1 保持基础配置中的微调方式和增强方式一致，只改变骨干网络。默认预训练权重会在首次启用训练时下载；训练前请确认设备和训练预算。多个随机种子用于评估波动，不能挑选某个种子作为唯一结果。

In [3]:
RUN_TRAINING = True

plans = []
for name in EXPERIMENTS.get("baseline_models", ["resnet18", "mobilenet_v2", "efficientnet_b0"]):
    for seed in EXPERIMENTS.get("seeds", [BASE.get("seed", 42)]):
        config = deepcopy(BASE)
        config["model"]["name"] = name
        config["seed"] = int(seed)
        run_id = f"e1_{name}_seed{seed}"
        plans.append({"run_id": run_id, "config": config})

show_table([{
    "run_id": p["run_id"],
    "model": p["config"]["model"]["name"],
    "finetuning": p["config"]["model"]["finetuning"],
    "augmentation": p["config"]["augmentation"],
    "seed": p["config"]["seed"],
    "epochs": p["config"]["training"]["epochs"],
    "lr": p["config"]["training"]["lr"],
} for p in plans])

run_id,model,finetuning,augmentation,seed,epochs,lr
e1_resnet18_seed42,resnet18,freeze,basic,42,10,0.001
e1_mobilenet_v2_seed42,mobilenet_v2,freeze,basic,42,10,0.001
e1_efficientnet_b0_seed42,efficientnet_b0,freeze,basic,42,10,0.001


## 2. 训练与验证

训练函数会将每次实验的完整配置、类别顺序、设备环境及最佳模型单独保存。已有同名实验目录时应使用新的实验编号，避免把多次实验混在一起。

In [4]:
if not RUN_TRAINING:
    print("尚未开始训练。确认固定清单和参数后，将 RUN_TRAINING 设为 True。")
elif not LABELS.get("classes"):
    print("请先完成 Notebook 01 的共同类别和固定划分。")
else:
    from src.train import run_experiment
    for plan in plans:
        run_dir = run_experiment(PROJECT_ROOT, plan["config"], plan["run_id"])
        print("已保存：", run_dir)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/qingjiarui/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 9.27MB/s]


e1_resnet18_seed42 | epoch 1: train_loss=0.9302, val_macro_f1=0.8084
e1_resnet18_seed42 | epoch 2: train_loss=0.5267, val_macro_f1=0.7797
e1_resnet18_seed42 | epoch 3: train_loss=0.4331, val_macro_f1=0.8463
e1_resnet18_seed42 | epoch 4: train_loss=0.3772, val_macro_f1=0.8650
e1_resnet18_seed42 | epoch 5: train_loss=0.3349, val_macro_f1=0.8643
e1_resnet18_seed42 | epoch 6: train_loss=0.3023, val_macro_f1=0.8677
e1_resnet18_seed42 | epoch 7: train_loss=0.2902, val_macro_f1=0.8684
e1_resnet18_seed42 | epoch 8: train_loss=0.2811, val_macro_f1=0.8666
e1_resnet18_seed42 | epoch 9: train_loss=0.2593, val_macro_f1=0.8650
e1_resnet18_seed42 | epoch 10: train_loss=0.2532, val_macro_f1=0.8563
已保存： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project/outputs/runs/e1_resnet18_seed42
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/qingjiarui/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:01<00:00, 12.5MB/s]


e1_mobilenet_v2_seed42 | epoch 1: train_loss=0.8029, val_macro_f1=0.8182
e1_mobilenet_v2_seed42 | epoch 2: train_loss=0.4106, val_macro_f1=0.8067
e1_mobilenet_v2_seed42 | epoch 3: train_loss=0.3384, val_macro_f1=0.8813
e1_mobilenet_v2_seed42 | epoch 4: train_loss=0.2962, val_macro_f1=0.8871
e1_mobilenet_v2_seed42 | epoch 5: train_loss=0.2643, val_macro_f1=0.8975
e1_mobilenet_v2_seed42 | epoch 6: train_loss=0.2454, val_macro_f1=0.9048
e1_mobilenet_v2_seed42 | epoch 7: train_loss=0.2413, val_macro_f1=0.9039
e1_mobilenet_v2_seed42 | epoch 8: train_loss=0.2262, val_macro_f1=0.8931
e1_mobilenet_v2_seed42 | epoch 9: train_loss=0.2107, val_macro_f1=0.9005
已保存： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project/outputs/runs/e1_mobilenet_v2_seed42
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/qingjiarui/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 12.4MB/s]


e1_efficientnet_b0_seed42 | epoch 1: train_loss=0.7373, val_macro_f1=0.8545
e1_efficientnet_b0_seed42 | epoch 2: train_loss=0.3955, val_macro_f1=0.8868
e1_efficientnet_b0_seed42 | epoch 3: train_loss=0.3151, val_macro_f1=0.8958
e1_efficientnet_b0_seed42 | epoch 4: train_loss=0.2726, val_macro_f1=0.8819
e1_efficientnet_b0_seed42 | epoch 5: train_loss=0.2386, val_macro_f1=0.9072
e1_efficientnet_b0_seed42 | epoch 6: train_loss=0.2174, val_macro_f1=0.9177
e1_efficientnet_b0_seed42 | epoch 7: train_loss=0.1980, val_macro_f1=0.9171
e1_efficientnet_b0_seed42 | epoch 8: train_loss=0.1929, val_macro_f1=0.9136
e1_efficientnet_b0_seed42 | epoch 9: train_loss=0.1854, val_macro_f1=0.9269
e1_efficientnet_b0_seed42 | epoch 10: train_loss=0.1703, val_macro_f1=0.9224
已保存： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project/outputs/runs/e1_efficientnet_b0_seed42


## 3. 汇总已有验证记录

此处只读取已生成的训练历史，不读取测试集。最佳轮次依据基础配置中的验证指标确定。最终选用的骨干名称写回 `configs/experiments.yaml` 的 `selected_backbone`。

In [5]:
validation_records = summarize_validation(plans)
show_table(validation_records)
if not validation_records:
    print("尚无训练记录。运行训练后，此处会显示最佳验证轮次及真实指标。")
print("当前选定骨干：", EXPERIMENTS.get("selected_backbone"))


run_id,recorded_epochs,monitor,best_epoch,val_loss,val_accuracy,val_macro_f1,total_training_seconds
e1_resnet18_seed42,10,macro_f1,7,0.3323686741259655,0.8810572687224669,0.8683988924335247,165.53610112500002
e1_mobilenet_v2_seed42,9,macro_f1,6,0.2691372820721849,0.9129955947136564,0.9047564542056625,181.86918437499997
e1_efficientnet_b0_seed42,10,macro_f1,9,0.19732479101235645,0.9361233480176211,0.9269425761028849,242.75108441699996


当前选定骨干： None


## 结论记录

- 统一训练条件与设备：待实际运行后填写。
- 各模型验证集表现、波动及资源开销：待填写。
- 选定骨干及选择依据（仅依据验证集）：待填写。
- 与下一阶段保持一致的设置：待填写。